# 常见的离散化方法概览

以下是常见的数值离散化方法的概览——用于把连续时间动力学
$$\dot x = f(x,u,t)$$
离散到时间步长为 $\Delta t$ 的递推形式。每种方法的阶数、稳定性与适用性不同：

- 显式欧拉（Explicit Euler）: 一阶，简单但条件稳定。
- 隐式欧拉（Implicit / Backward Euler）: 一阶，A-稳定，刚性系统常用。
- 中点法 / 二阶Runge-Kutta（RK2）: 二阶，精度较欧拉高。
- 四阶Runge-Kutta（RK4）: 四阶，常用的高精度方法。
- Trapezoidal / Crank–Nicolson: 二阶，对称且具有更好能量行为。
- 多步法（如BDF）与伪谱方法（Legendre / Chebyshev）: 适用于高精度或稀疏网格。

下面给出每种方法的公式和快速说明（下一单元详细）。（此单元为第 1 单元）

## 离散化方法与公式（详细）

1) 显式欧拉（Explicit Euler）

$$
x_{k+1} = x_k + f(x_k,u_k,t_k)\,\Delta t.
$$
- 精度：一阶。
- 优点：简单、计算便宜。
- 缺点：仅条件稳定（步长受限）。

2) 隐式欧拉（Backward / Implicit Euler）

$$
x_{k+1} = x_k + f(x_{k+1},u_{k+1},t_{k+1})\,\Delta t.
$$
- 精度：一阶。
- 优点：A-稳定，适合刚性系统。
- 缺点：需要求解隐式方程（通常用牛顿迭代）。

3) Trapezoidal / Crank–Nicolson（中点-梯形）

$$
x_{k+1} = x_k + \tfrac{\Delta t}{2} \big(f(x_k,u_k,t_k) + f(x_{k+1},u_{k+1},t_{k+1})\big).
$$
- 精度：二阶。对某些物理系统具有良好守恒性质。

4) 二阶 Runge–Kutta（中点法 / RK2）

中点法形式：
$$
\begin{aligned}
&k_1 = f(x_k,u_k,t_k),\\
&k_2 = f(x_k + \tfrac{\Delta t}{2}k_1,\; u_k,\; t_k+\tfrac{\Delta t}{2}),\\
&x_{k+1} = x_k + \Delta t\, k_2.
\end{aligned}
$$
- 精度：二阶。

5) 四阶 Runge–Kutta（RK4）

$$
\begin{aligned}
&k_1 = f(x_k,u_k,t_k),\\
&k_2 = f(x_k + \tfrac{\Delta t}{2}k_1,\; u_k,\; t_k+\tfrac{\Delta t}{2}),\\
&k_3 = f(x_k + \tfrac{\Delta t}{2}k_2,\; u_k,\; t_k+\tfrac{\Delta t}{2}),\\
&k_4 = f(x_k + \Delta t\, k_3,\; u_k,\; t_k+\Delta t),\\
&x_{k+1} = x_k + \tfrac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4).
\end{aligned}
$$
- 精度：四阶，常用的通用方法。

6) 伪谱与高阶方法（简述）

- 伪谱方法（如 Gauss/Lobatto 伪谱）通过在全域上用高阶多项式近似状态，适合需要高精度的最优控制问题（常用于无网格直接法）。

（此单元为第 2 单元）

# 车辆模型离散化（仅列两种实现：前向欧拉 与 隐式中点（基于积分中值定理））

考虑运动学双轮模型（kinematic bicycle model），状态为
$$x=[x,\ y,\ \theta,\ v]^T$$
控制为
$$u=[a,\ \delta]^T$$
车轮轴距为 $L$，连续动力学为：

$$
\dot x = v\cos\theta, \quad
\dot y = v\sin\theta, \quad
\dot \theta = \frac{v}{L}\tan\delta, \quad
\dot v = a.
$$

下面分别给出两种常用离散化与对应**解析雅可比（A,B）**：

---

### 1) 前向欧拉（Forward Euler）

更新：
$$
x_{k+1} = x_k + f(x_k,u_k)\,\Delta t.
$$
雅可比（解析）：
$$
A = I + F_x(x_k,u_k)\,\Delta t,
\qquad
B = F_u(x_k,u_k)\,\Delta t.
$$
（其中 $F_x, F_u$ 为连续动力学对 $x,u$ 的偏导。）
$$
x_{k+1} = Ax_k + Bu_k.
$$
---

### 2) 隐式中点（基于积分中值定理）

根据积分中值定理，存在点 $\xi\in[t_k,t_{k+1}]$ 使得
$$x_{k+1}=x_k + \Delta t\, f(x(\xi), u(\xi)).$$
取近似 $x(\xi)\approx\tfrac{1}{2}(x_k + x_{k+1})$ 且 $u(\xi)\approx u_k$，得到**隐式中点规则**：

$$
x_{k+1} = x_k + \Delta t\, f\Big(\tfrac{x_k+x_{k+1}}{2},\; u_k\Big).
$$

定义 $m=\tfrac{1}{2}(x_k+x_{k+1})$，对上式作隐式微分可得解析雅可比（将 $x_{k+1}$ 视为 $x_k,u_k$ 的函数）：

$$
\big(I - \tfrac{\Delta t}{2}F_x(m,u_k)\big)\;A = I + \tfrac{\Delta t}{2}F_x(m,u_k),
$$
$$
\big(I - \tfrac{\Delta t}{2}F_x(m,u_k)\big)\;B = \Delta t\,F_u(m,u_k).
$$

因此
$$
A = \Big(I - \tfrac{\Delta t}{2}F_x(m,u_k)\Big)^{-1} \Big(I + \tfrac{\Delta t}{2}F_x(m,u_k)\Big),
\quad
B = \Big(I - \tfrac{\Delta t}{2}F_x(m,u_k)\Big)^{-1} \Delta t\,F_u(m,u_k).
$$

提示：该方法为二阶精度，且显式地使用了积分中值定理的中点近似；$m$ 依赖于 $x_{k+1}$，因此在实现上需对 $x_{k+1}$ 做隐式求解（例如用牛顿或固定点迭代）。

# 离散 LQR 推导（整理 J，区分第 N 步与其他步）

下面把离散 LQR 的推导整理清楚，明确第 $N$ 步（终端）与其他步的状态方程与协态方程的不同。

## 问题设置
离散线性系统：
$$
x_{k+1} = A x_k + B u_k,
$$
代价函数（把步长吸收在矩阵中或直接使用离散化后的权重）：
$$
J = \sum_{k=0}^{N-1} \left( \tfrac{1}{2} x_k^T Q x_k + \tfrac{1}{2} u_k^T R u_k \right) + \tfrac{1}{2} x_N^T P_N x_N,
$$
其中 $Q\succeq0,\ R\succ0,\ P_N$ 为终端权重。

## 增广 Lagrangian（引入协态作为动态等式约束的乘子）
对动态约束 $x_{k+1}-A x_k - B u_k = 0$ 引入拉格朗日乘子 $\lambda_{k+1}$，构造：
$$
\mathcal{L} = J + \sum_{k=0}^{N-1} \lambda_{k+1}^T (A x_k + B u_k - x_{k+1}).
$$

对 $x_k,u_k,x_N$ 变分并令驻点条件成立得到必要条件（KKT/庞特金条件）。下面把第 $N$ 步与其他步区分开写。

---

## 终端条件（第 $N$ 步）
终端对 $x_N$ 的驻点条件：
$$
\frac{\partial \mathcal{L}}{\partial x_N} = P_N x_N - \lambda_N = 0 \quad\Rightarrow\quad \boxed{\lambda_N = P_N x_N.}
$$
这里没有 $u_N$，只有终端协态边界条件。

---

## 对于 $k=0,1,\dots,N-1$ 的一般步（状态方程与协态方程）
1) 状态方程（原始约束）：
$$
\boxed{x_{k+1} = A x_k + B u_k.}
$$
2) 控制驻点（对 $u_k$ 求导）：
$$
\frac{\partial \mathcal{L}}{\partial u_k} = R u_k + B^T \lambda_{k+1} = 0 \quad\Rightarrow\quad \boxed{R u_k + B^T \lambda_{k+1} = 0.}
$$
3) 状态驻点 / 协态更新（对 $x_k$ 求导）：
$$
\frac{\partial \mathcal{L}}{\partial x_k} = Q x_k + A^T \lambda_{k+1} - \lambda_k = 0
\quad\Rightarrow\quad \boxed{\lambda_k = Q x_k + A^T \lambda_{k+1}.}
$$
这三条方程在每个 $k=0,\dots,N-1$ 都成立，其中协态递推是**后向**的（由 $\lambda_N$ 开始反向计算）。

---

## 从协态关系到 Riccati 方程（假设二次形式）
假设存在矩阵 $P_k$ 使得 $\lambda_k = P_k x_k$（这在二次代价且线性系统时成立）。代入协态更新：
$$
P_k x_k = Q x_k + A^T P_{k+1} x_{k+1}.
$$
用状态方程 $x_{k+1} = A x_k + B u_k$ 并把 $u_k$ 用驻点条件表示：
$$
u_k = -R^{-1} B^T \lambda_{k+1} = -R^{-1} B^T P_{k+1} x_{k+1}.
$$
将 $u_k$ 消去并整理（省略中间代数步骤），可得到矩阵递推：
$$
\boxed{P_k = Q + A^T P_{k+1} A - A^T P_{k+1} B (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A,}
$$
终端条件： $P_N$ 给定。
同时获得最优状态反馈：
$$
\boxed{u_k = -K_k x_k,\qquad K_k = (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A.}
$$

---

## 小结（要点）
- **终端步（$k=N$）**只给出协态边界 $\lambda_N = P_N x_N$；没有控制方程。  
- **中间步（$k=0,...,N-1$）**满足状态前向、协态后向和控制的驻点条件，三者联合构成 TPBVP（two-point boundary value problem）。  
- 假设 $\lambda_k$ 与 $x_k$ 的线性关系后，得到 Riccati 的后向递推和线性反馈律，这也就是离散 LQR 的标准解法。


## 动态规划（价值函数 V 与动作值函数 Q）的推导

下面以动态规划的 V/Q 表达来推导离散 LQR，步骤清晰，便于与哈密顿/协态方法对应。

1) 定义价值函数与动作值函数：
$$
V_k(x_k)=\min_{u_k,\dots,u_{N-1}}\;\sum_{i=k}^{N-1}\left(\tfrac12 x_i^T Q x_i + \tfrac12 u_i^T R u_i\right)+\tfrac12 x_N^T P_N x_N.
$$
动作值函数（Q 函数）：
$$
Q_k(x_k,u_k)=\tfrac12 x_k^T Q x_k + \tfrac12 u_k^T R u_k + V_{k+1}(x_{k+1}),\quad x_{k+1}=A x_k + B u_k.
$$
贝尔曼方程：$$V_k(x_k)=\min_{u_k} Q_k(x_k,u_k).$$

2) 假设二次形式（LQR 特例）：若$V_{k+1}(x)=\tfrac12 x^T P_{k+1} x$，代入得
$$
Q_k(x,u)=\tfrac12 x^T Q x + \tfrac12 u^T R u + \tfrac12 (A x + B u)^T P_{k+1}(A x + B u).
$$
展开并收集关于 $x,u$ 的项：
$$
Q_{xx} = Q + A^T P_{k+1} A,
\qquad Q_{uu} = R + B^T P_{k+1} B,
\qquad Q_{ux} = B^T P_{k+1} A.
$$

3) 对 $u$ 求极小化（解析解）：
$$\frac{\partial Q_k}{\partial u}=0 \Rightarrow Q_{uu} u + Q_{ux} x = 0, $$
因此得到最优反馈
$$
u^*_k = -Q_{uu}^{-1} Q_{ux} x_k = -K_k x_k,
\qquad K_k = (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A.
$$

4) 代回得到 Riccati 递推：
$$
P_k = Q + A^T P_{k+1} A - A^T P_{k+1} B (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A,
$$
终端条件：$P_N$ 已知（由终端代价给出）。

5) 价值函数与协态的对应：$\lambda_k = \partial V_k/\partial x_k = P_k x_k$，因此动态规划的梯度形式与哈密顿/协态方法完全一致。

---


# 并列推导：动态规划 (V/Q) 与 庞特金 (PMP) 的详细比较

下面把你给出的两条路径**逐步并列推导**并详细说明“把什么代入到什么里”的关键差异，最后列出统一的闭式结果。

---

## 1) 动态规划（V / Q 法）

- 已知假设：$V_{k+1}(x)=\tfrac12 x^\top P_{k+1} x$。

- 写动作值函数：
  $$
  Q_k(x,u)=\tfrac12 x^\top Q x + \tfrac12 u^\top R u + \tfrac12 (A x + B u)^\top P_{k+1} (A x + B u).
  $$

- 展开并收集关于 $x,u$ 的二次项：
  $$
  Q_{xx} = Q + A^\top P_{k+1} A,\quad Q_{uu} = R + B^\top P_{k+1} B,\quad Q_{ux} = B^\top P_{k+1} A.
  $$

- 对 $u$ 求极小化（解析求导）：
  $$
  0 = \partial_u Q_k = Q_{uu} u + Q_{ux} x \quad\Rightarrow\quad u^\star = - Q_{uu}^{-1} Q_{ux} x.
  $$
  即
  $$
  u_k^\star = - (R + B^\top P_{k+1} B)^{-1} B^\top P_{k+1} A\, x_k.
  $$

- **代入位置与消元**：把 $u^\star$ 代回 $Q_k(x,u)$，得到
  $$
  V_k(x) = Q_k(x,u^\star) = \tfrac12 x^\top P_k x.
  $$
  比对 $x^\top(\cdot)x$ 的二次系数即可得到 Riccati 递推（见结论）。

- **关键点**：把 $P_{k+1}$（即 $V_{k+1}$）代入 $Q_k$，然后对 $u$ 直接极小化并消去 $u$，最后从剩余二次项中读出 $P_k$。

---

## 2) 庞特金 / 哈密顿（PMP）

- 控制驻点给出：
  $$
  0 = \partial_u H = R u + B^\top \lambda_{k+1} \quad\Rightarrow\quad u^\star = -R^{-1} B^\top \lambda_{k+1}.
  $$

- 协态后向递推：
  $$
  \lambda_k = Q x_k + A^\top \lambda_{k+1}.
  $$

- 用线性假设代入：设 $\lambda_k = P_k x_k$，并写 $\lambda_{k+1}=P_{k+1} x_{k+1}=P_{k+1}(A x_k + B u_k)$，因此
  $$
  P_k x_k = Q x_k + A^\top P_{k+1} A x_k + A^\top P_{k+1} B u_k.
  $$

- 把控制驻点代入（先隐式）并解出显式 $u_k$：
  $$
  u_k = -R^{-1} B^\top P_{k+1} (A x_k + B u_k).
  $$
  整理得
  $$
  (R + B^\top P_{k+1} B) u_k = - B^\top P_{k+1} A x_k,
  $$
  从而
  $$
  u_k = - (R + B^\top P_{k+1} B)^{-1} B^\top P_{k+1} A\, x_k.
  $$

- **代回协态并比对系数**：将该 $u_k$ 代回协态递推，匹配 $x_k$ 的二次项系数，得到 Riccati 递推（见结论）。

- **关键点**：先用驻点将 $u$ 表示为 $\lambda_{k+1}$，再用 $\lambda_{k+1}=P_{k+1}x_{k+1}$ 代入协态方程，解出 $u$（可能先隐式后显式），最终消元得到 $P_k$。

---

## 核心结论
总结起来核心点是通过递推方程对比，等式两边系数相等得到Riccati，
协态方程是最优值函数对应的方程的导数形式，即：
$$
V_k(x) = \tfrac12 x^\top P_k x = Q_k(x,u^\star)=\min_{u_k} \tfrac12 x_k^T Q x_k + \tfrac12 u_k^T R u_k + V_{k+1}(x_{k+1}).
$$
等式两边对$x_k$求导:
$$
\lambda_k =P_k x_k = Q x_k + A^\top \lambda_{k+1}.
$$
然后两个方程分别带入$u^*$，得到riccati方程

两条路径代入/消元顺序不同，但代数上等价，最终得到相同的闭式控制律与 Riccati 递推：

$$
u_k^\star = - (R + B^\top P_{k+1} B)^{-1} B^\top P_{k+1} A\, x_k,
$$

$$
P_k = Q + A^\top P_{k+1} A - A^\top P_{k+1} B (R + B^\top P_{k+1} B)^{-1} B^\top P_{k+1} A.
$$

---
